# MARKETPLACE 360 · Data Factory — Semana 9
## Generador de **10 tablas realistas** en CSV

**Sin scripts SQL, sin BULK INSERT y sin conexión a SQL Server.** Este notebook genera 10 archivos `.csv` con datos ficticios coherentes y los empaqueta en un `.zip`.

**Tablas:** CATEGORIA, MARCA, TIENDA, PRODUCTO, PRECIO_TIENDA, CLIENTE, PEDIDO, DETALLE_PEDIDO, CATALOGO_WEB y CATALOGO_ERP.

**Relaciones:** categoría→producto; marca→producto; producto/tienda→precio; cliente→pedido; pedido/producto→detalle. Los catálogos WEB y ERP incluyen coincidencias y diferencias para practicar FULL JOIN.

**Uso:** ejecuta las celdas en orden. En Google Colab, al final se descargará el ZIP; en Jupyter quedará guardado en la carpeta de trabajo. Los CSV usan UTF-8 con BOM para facilitar la apertura en Excel, y separador coma.


In [ ]:
# 1. Configuración: modifica únicamente estos valores
MODO = "COMPLETO"  # "LIGERO" o "COMPLETO"
SEMILLA = 202609
CARPETA_SALIDA = "MARKETPLACE_S9_DATOS"

import csv, random, zipfile, math
from pathlib import Path
from datetime import date, timedelta
from collections import Counter

rng = random.Random(SEMILLA)
out = Path(CARPETA_SALIDA)
out.mkdir(parents=True, exist_ok=True)

if MODO.upper() == 'COMPLETO':
    N_PRODUCTOS, N_CLIENTES, N_PEDIDOS, N_PRECIOS, N_DETALLES = 50000, 10000, 30000, 100000, 100000
    N_CATEGORIAS, N_MARCAS, N_TIENDAS = 30, 100, 50
elif MODO.upper() == 'LIGERO':
    N_PRODUCTOS, N_CLIENTES, N_PEDIDOS, N_PRECIOS, N_DETALLES = 1500, 1000, 1500, 2500, 3500
    N_CATEGORIAS, N_MARCAS, N_TIENDAS = 15, 25, 12
else:
    raise ValueError('MODO debe ser LIGERO o COMPLETO')

ARCHIVOS = {}
def guardar(nombre, columnas, filas):
    ruta = out / f'{nombre}.csv'
    with ruta.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.writer(f)
        w.writerow(columnas)
        w.writerows(filas)
    ARCHIVOS[nombre] = {'ruta':ruta, 'filas':len(filas), 'columnas':len(columnas)}
    print(f'{nombre:17} {len(filas):>9,} filas  | {len(columnas):>2} columnas')

def dinero(v): return f'{v:.2f}'
def fecha_aleatoria(inicio, dias): return (inicio + timedelta(days=rng.randrange(dias))).isoformat()
print('Modo:', MODO, '| Semilla:', SEMILLA)


In [ ]:
# 2. Catálogos maestros
nombres_categorias = [
    'Laptops','Tablets','Celulares','Monitores','Teclados','Mouse','Audífonos','Parlantes',
    'Impresoras','Routers','Discos SSD','Memorias RAM','Webcams','Cámaras','Smartwatch',
    'Televisores','Proyectores','Sillas Gamer','Escritorios','Mochilas','Cargadores',
    'Cables','Adaptadores','Consolas','Videojuegos','Microfonos','Drones','Accesorios',
    'Componentes PC','Equipos de Red'][:N_CATEGORIAS]
marcas_base = ['AndesTech','InkaDigital','NovaByte','PumaTech','SolucionesX','Altura','Qori','Ayni',
               'Inti','Orion','Vertex','Nimbus','Pulsar','Kuntur','LimaPro','Pacifica','Arco','Misti',
               'Nexo','Andromeda']
marcas = [f'{marcas_base[(i-1)%len(marcas_base)]} {((i-1)//len(marcas_base))+1:02d}' for i in range(1,N_MARCAS+1)]
ciudades = ['Lima','Arequipa','Trujillo','Chiclayo','Piura','Cusco','Huancayo','Iquitos','Tacna','Puno','Huánuco','Ica']
distritos = ['Miraflores','San Isidro','Surco','Los Olivos','San Miguel','Cercado','La Molina','Breña','Jesús María','Callao']
categorias = [(i,nombres_categorias[i-1],f'Equipos y accesorios de {nombres_categorias[i-1].lower()}',int(i%7!=0)) for i in range(1,N_CATEGORIAS+1)]
marcas_rows = [(i,marcas[i-1],rng.choice(['Perú','China','Estados Unidos','Corea del Sur','Japón','Alemania']),int(i%11!=0)) for i in range(1,N_MARCAS+1)]
tiendas = [(i,f'Tienda {ciudades[(i-1)%len(ciudades)]} {i:02d}',ciudades[(i-1)%len(ciudades)],rng.choice(distritos),f'Av. Comercio {100+i*17}',rng.choice(['Centro comercial','Local propio','Almacén']),int(i%13!=0)) for i in range(1,N_TIENDAS+1)]
guardar('CATEGORIA',['IDCategoria','NombreCategoria','Descripcion','Activo'],categorias)
guardar('MARCA',['IDMarca','NombreMarca','PaisOrigen','Activo'],marcas_rows)
guardar('TIENDA',['IDTienda','NombreTienda','Ciudad','Distrito','Direccion','TipoTienda','Activo'],tiendas)


In [ ]:
# 3. Productos: atributos, precios y stock coherentes
adjetivos = ['Pro','Plus','Max','Lite','Air','Ultra','Core','Prime','Go','Elite','Neo','Flex']
colores = ['Negro','Plata','Blanco','Azul','Gris','Grafito']
productos=[]
for i in range(1,N_PRODUCTOS+1):
    categoria_id = rng.randint(1, N_CATEGORIAS-3)  # reservar categorías sin productos
    marca_id = rng.randint(1,N_MARCAS)
    base = rng.uniform(35, 7500)
    costo = round(base*rng.uniform(.57,.82),2)
    precio = round(base,2)
    stock = rng.choice([0,0,1,2,4,7,12,20,45,100])
    productos.append((i, f'{nombres_categorias[categoria_id-1]} {rng.choice(adjetivos)} {i:05d}',
       categoria_id,marca_id,f'Modelo {rng.choice(adjetivos)} para uso personal y empresarial',
       f'SKU-{i:07d}',rng.choice(colores),dinero(costo),dinero(precio),stock,
       fecha_aleatoria(date(2022,1,1),1700),int(i%29!=0)))
guardar('PRODUCTO',['IDProducto','NombreProducto','IDCategoria','IDMarca','Descripcion','SKU','Color','Costo','PrecioLista','StockGeneral','FechaAlta','Activo'],productos)


In [ ]:
# 4. Precios por tienda: cada par producto-tienda es único
# Reservar las últimas 2 tiendas sin precios para ejercicios LEFT/RIGHT JOIN
max_tienda_precio=max(1,N_TIENDAS-2)
pares=set()
while len(pares)<N_PRECIOS:
    pares.add((rng.randint(1,N_PRODUCTOS),rng.randint(1,max_tienda_precio)))
precios=[]
for idx,(prod,tienda) in enumerate(sorted(pares),1):
    precio_base=float(productos[prod-1][8])
    precio=round(precio_base*rng.uniform(.86,1.14),2)
    precios.append((idx,prod,tienda,dinero(precio),rng.choice([0,0,2,5,10,20,50]),
                    fecha_aleatoria(date(2025,1,1),600),rng.choice(['Regular','Promoción','Liquidación'])))
guardar('PRECIO_TIENDA',['IDPrecioTienda','IDProducto','IDTienda','Precio','StockTienda','FechaActualizacion','TipoPrecio'],precios)


In [ ]:
# 5. Clientes, pedidos y detalle de ventas
nombres=['Ana','Luis','María','Carlos','Rosa','José','Elena','Diego','Lucía','Miguel','Valeria','Jorge','Camila','Pedro','Sofía','Andrés']
apellidos=['García','Quispe','Flores','Torres','Rojas','Vargas','Huamán','Mendoza','Castillo','Ramírez','Salazar','Paredes']
clientes=[]
for i in range(1,N_CLIENTES+1):
    nom=rng.choice(nombres); ape=rng.choice(apellidos)
    clientes.append((i,nom,ape,f'cliente{i:06d}@correo-ejemplo.pe',f'9{i%100000000:08d}',
                     rng.choice(ciudades),rng.choice(distritos),fecha_aleatoria(date(2021,1,1),1700),
                     rng.choice(['Minorista','Empresa','Premium'])))
guardar('CLIENTE',['IDCliente','Nombre','Apellido','Correo','Telefono','Ciudad','Distrito','FechaRegistro','Segmento'],clientes)
# Reservar los últimos clientes sin pedidos, para análisis de ausencia
max_cliente_pedido=max(1,N_CLIENTES-150)
pedidos=[]
for i in range(1,N_PEDIDOS+1):
    pedidos.append((i,rng.randint(1,max_cliente_pedido),rng.randint(1,N_TIENDAS),
                    fecha_aleatoria(date(2025,1,1),600),rng.choice(['Entregado','Pendiente','Enviado','Cancelado']),
                    rng.choice(['Yape','Tarjeta','Transferencia','Efectivo']),
                    rng.choice(['Web','App','Tienda']),rng.choice(ciudades)))
guardar('PEDIDO',['IDPedido','IDCliente','IDTienda','FechaPedido','Estado','MetodoPago','CanalVenta','CiudadEntrega'],pedidos)
# Al menos un detalle por pedido, luego completar hasta el volumen objetivo
detalles=[]
for i in range(1,N_DETALLES+1):
    pedido_id=i if i<=N_PEDIDOS else rng.randint(1,N_PEDIDOS)
    producto_id=rng.randint(1,N_PRODUCTOS)
    cantidad=rng.randint(1,5)
    unitario=round(float(productos[producto_id-1][8])*rng.uniform(.90,1.06),2)
    descuento=round(unitario*cantidad*rng.choice([0,0,0,.05,.10]),2)
    subtotal=round(unitario*cantidad-descuento,2)
    detalles.append((i,pedido_id,producto_id,cantidad,dinero(unitario),dinero(descuento),dinero(subtotal)))
guardar('DETALLE_PEDIDO',['IDDetallePedido','IDPedido','IDProducto','Cantidad','PrecioUnitario','Descuento','Subtotal'],detalles)


In [ ]:
# 6. Catálogos WEB y ERP: coincidencias y registros exclusivos
# Códigos 1-400: ambos; 401-450: solo WEB; 451-500: solo ERP
web=[]; erp=[]
for i in range(1,451):
    web.append((f'CAT-{i:05d}',f'Artículo digital {i:05d}',rng.choice(['Publicado','Borrador']),
                dinero(rng.uniform(50,3000)),fecha_aleatoria(date(2025,1,1),600)))
for i in list(range(1,401))+list(range(451,501)):
    erp.append((f'CAT-{i:05d}',f'Artículo inventario {i:05d}',rng.choice(['Disponible','Revisión']),
                rng.randint(0,300),fecha_aleatoria(date(2025,1,1),600)))
guardar('CATALOGO_WEB',['Codigo','Descripcion','EstadoWeb','PrecioWeb','FechaPublicacion'],web)
guardar('CATALOGO_ERP',['Codigo','Descripcion','EstadoERP','StockERP','FechaRegistro'],erp)


In [ ]:
# 7. Validaciones de calidad
assert len({r[0] for r in productos})==N_PRODUCTOS
assert len({(r[1],r[2]) for r in precios})==N_PRECIOS
assert all(1<=r[2]<=N_CATEGORIAS for r in productos)
assert all(1<=r[3]<=N_MARCAS for r in productos)
assert all(1<=r[1]<=N_PRODUCTOS and 1<=r[2]<=N_TIENDAS for r in precios)
assert all(1<=r[1]<=N_CLIENTES and 1<=r[2]<=N_TIENDAS for r in pedidos)
assert all(1<=r[1]<=N_PEDIDOS and 1<=r[2]<=N_PRODUCTOS for r in detalles)
assert len({r[0] for r in web}&{r[0] for r in erp})==400
assert len({r[0] for r in web}-{r[0] for r in erp})==50
assert len({r[0] for r in erp}-{r[0] for r in web})==50
assert all(r[1]<=max_cliente_pedido for r in pedidos)
print('VALIDACIÓN CORRECTA: claves, relaciones y datos para JOIN.')
print('Clientes sin pedidos (garantizados):',N_CLIENTES-max_cliente_pedido)
print('Tiendas sin precios (garantizadas):',N_TIENDAS-max_tienda_precio)
print('Categorías sin productos (garantizadas):',3)
print('WEB/ERP: 400 coincidencias, 50 solo WEB y 50 solo ERP')
print('TOTAL DE FILAS:',f'{sum(x["filas"] for x in ARCHIVOS.values()):,}')


In [ ]:
# 8. Crear ZIP con SOLO los 10 CSV (no se generan scripts SQL)
zip_path=Path('MARKETPLACE_S9_10_TABLAS_CSV.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for nombre,info in ARCHIVOS.items():
        z.write(info['ruta'],arcname=info['ruta'].name)
print('ZIP creado:',zip_path.resolve())
print('Contiene',len(ARCHIVOS),'CSV:')
for nombre,info in ARCHIVOS.items():
    print(f'  {nombre}.csv — {info["filas"]:,} filas, {info["columnas"]} columnas')
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('En Jupyter, descarga el ZIP desde la carpeta del notebook.')


## ¿Cómo utilizo los archivos?

1. Descarga `MARKETPLACE_S9_10_TABLAS_CSV.zip` y descomprímelo.
2. Obtendrás **10 archivos CSV**, uno por tabla, con encabezados claros.
3. Puedes abrirlos en Excel o importarlos a SQL Server con **Import Flat File** / el asistente de importación, configurando tipos de datos y relaciones según la guía del laboratorio. **Este notebook no crea tablas directamente dentro de SQL Server.**
4. Orden recomendado de importación si ya existen claves foráneas: **CATEGORIA, MARCA, TIENDA, CLIENTE, PRODUCTO, PEDIDO, PRECIO_TIENDA, DETALLE_PEDIDO, CATALOGO_WEB, CATALOGO_ERP**.

**Nota:** los datos son sintéticos y los correos usan un dominio de ejemplo. No contienen datos personales reales.
